In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.44.2 einops timm

import os
import cv2
import json
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

# ================= THE CORRECTED HUGGING FACE BYPASS PATCH =================
import transformers.dynamic_module_utils as dyn_utils
original_check = dyn_utils.check_imports

def custom_check_imports(filename):
    try:
        return original_check(filename)
    except ImportError as e:
        if "flash_attn" in str(e):
            return dyn_utils.get_relative_imports(filename)
        else:
            raise e
dyn_utils.check_imports = custom_check_imports
# =========================================================================

# ================= CONFIGURATION =================
INPUT_VIDEO = "/kaggle/input/datasets/gonoszgonosz/rat-test-video/test.mp4" 

# Outputs
OUTPUT_VIDEO = "/kaggle/working/Florence2_Video_Output.mp4"
OUTPUT_MASK_DIR = "/kaggle/working/florence2_outliers"
OUTPUT_JSON = "/kaggle/working/florence2_outlier_logs.json"

MODEL_ID = "microsoft/Florence-2-large"
TASK_PROMPT = "<REFERRING_EXPRESSION_SEGMENTATION>"
TEXT_INPUT = " the rat"  # Updated prompt!

device = "cuda" if torch.cuda.is_available() else "cpu"
# =================================================

def apply_overlay(image, mask, color=(0, 255, 255), alpha=0.5):
    """Blends a solid color over the masked region (Yellow for Florence)."""
    overlay = np.full_like(image, color)
    blended = cv2.addWeighted(image, 1 - alpha, overlay, alpha, 0)
    res = image.copy()
    res[mask == 1] = blended[mask == 1]
    return res

def main():
    os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
    
    print("--- LOADING FLORENCE-2 VLM ---")
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True
    ).to(device).eval()

    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        print(f"Error: Could not open video {INPUT_VIDEO}")
        return

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

    outlier_log = []
    rolling_area_history = []
    
    print(f"--- STARTING FULL VIDEO INFERENCE & OUTLIER LOGGING ({total_frames} FRAMES) ---")
    pbar = tqdm(total=total_frames)
    
    prompt = TASK_PROMPT + TEXT_INPUT
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
            
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(image_rgb)
        
        # 1. Florence-2 Inference
        inputs = processor(text=prompt, images=pil_image, return_tensors="pt").to(device, torch.float16)
        
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=2048, 
                do_sample=False,
                num_beams=3
            )
            
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
        parsed_answer = processor.post_process_generation(generated_text, task=TASK_PROMPT, image_size=(w, h))
        
        # 2. Extract Data & Draw Masks
        pred_mask = np.zeros((h, w), dtype=np.uint8)
        polygons_dict = parsed_answer.get(TASK_PROMPT, {})
        polygons_list = polygons_dict.get('polygons', polygons_dict.get('Polygons', []))
        
        vertex_count = 0
        num_polygons = 0
        
        for obj_polys in polygons_list:
            for poly in obj_polys:
                if len(poly) >= 6: 
                    num_polygons += 1
                    vertex_count += len(poly) // 2 
                    poly_np = np.array(poly).reshape(-1, 2).astype(np.int32)
                    
                    # Draw solid mask for calculations (1s instead of 255s)
                    cv2.fillPoly(pred_mask, [poly_np], 1)
                    
                    # Draw a stark outline to show the VLM's anchor points on the video
                    cv2.polylines(frame, [poly_np], isClosed=True, color=(0, 255, 255), thickness=2)
        
        # 3. Apply Visual Overlay for the Video
        res_frame = apply_overlay(frame, pred_mask)
        cv2.putText(res_frame, "Florence-2: Analyzing Boundaries", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        out.write(res_frame)

        # 4. MATHEMATICAL OUTLIER DETECTION LOGIC
        mask_area = np.sum(pred_mask == 1)
        is_outlier = False
        reasons = []

        # Trigger A: Blocky/Vertex Starvation
        if 0 < vertex_count < 18:
            is_outlier = True
            reasons.append(f"Low Vertex Count ({vertex_count})")
            
        # Trigger B: Fragmented/Disconnected masks
        if num_polygons > 1:
            is_outlier = True
            reasons.append(f"Fragmented Mask ({num_polygons} polygons)")

        # Trigger C: Area Anomaly (Sudden jump in size)
        if len(rolling_area_history) > 10:
            avg_area = np.mean(rolling_area_history[-15:])
            if mask_area > (avg_area * 1.8):
                is_outlier = True
                reasons.append("Area Anomaly (Bigger than average)")
        
        # Update rolling area if it found the rat
        if mask_area > 0:
            rolling_area_history.append(mask_area)
        
        # 5. SAVE EVIDENCE IF OUTLIER DETECTED
        if is_outlier:
            # Convert 1s to 255s so the saved PNG is visibly white, not black
            bitmask_img = pred_mask * 255 
            mask_filename = os.path.join(OUTPUT_MASK_DIR, f"frame_{frame_idx:04d}_outlier.png")
            cv2.imwrite(mask_filename, bitmask_img)
            
            # Save mathematical evidence
            outlier_log.append({
                "frame_index": frame_idx,
                "reasons": reasons,
                "total_vertices": vertex_count,
                "mask_area_pixels": float(mask_area),
                "parsed_coordinates": polygons_list,
                "raw_model_text": generated_text
            })
        
        frame_idx += 1
        pbar.update(1)

    cap.release()
    out.release()
    pbar.close()

    # Save the JSON log
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(outlier_log, f, indent=4)

    print("\n" + "="*50)
    print(" INFERENCE AND OUTLIER LOGGING COMPLETE")
    print("="*50)
    print(f" Video saved to    : {OUTPUT_VIDEO}")
    print(f" Found Outliers    : {len(outlier_log)} frames")
    print(f" Evidence saved to : {OUTPUT_MASK_DIR}")
    print(f" JSON Log saved to : {OUTPUT_JSON}")
    print("="*50)

if __name__ == "__main__":
    main()